# Predicting Next-Day Air Quality (AQI) Across Ohio Counties

CSCI 6951 poster project: descriptive statistics and predictive models on daily county AQI plus weather, January 2022 to May 2026.

**How to run:** choose **Runtime > Run all**. The first cell asks you to upload `AQI_Ohio_2022_2026_with_weather.xlsx`. A normal CPU runtime is enough (no GPU needed). Most of the run time goes to the hyperparameter-tuning step (about 13 of the roughly 15 minutes in my test run); the rest takes a couple of minutes. Keep the tab open while it runs. The last cell downloads a zip with `results.json` and all figures.

## 1. Setup

In [ ]:
# 1. Setup: libraries, plot style and the data file
import os, warnings
import json, numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

OUT = {}
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 22, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.titleweight": "bold"})
RED, BLK, GRY = "#C8102E", "#1A1A1A", "#8C8C8C"
warnings.filterwarnings("ignore", message="A worker stopped while some jobs were given to the executor")   # harmless joblib notice

DATA = "AQI_Ohio_2022_2026_with_weather.xlsx"
if not os.path.exists(DATA):
    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError(f"Put {DATA} in the same folder as this notebook.")
    print(f"Choose {DATA} from your computer ...")
    uploaded = files.upload()          # opens a file picker; the file is saved in the Colab session
    DATA = next(iter(uploaded))        # use whatever name the uploaded file ended up with
print("Using data file:", DATA)

## 2. Load the data

Reads the Excel file, drops duplicate county-days and adds a season column (Dec-Feb winter, Mar-May spring, Jun-Aug summer, otherwise fall).

In [ ]:
d = pd.read_excel(DATA).rename(columns={"county Name": "County"})
print("Rows in file:", len(d), "| duplicate county-days:", int(d.duplicated(["County", "Date"]).sum()),
      "| missing cells:", int(d.isna().sum().sum()))
d = d.drop_duplicates(["County", "Date"]).sort_values(["County", "Date"]).reset_index(drop=True)
OUT["rows"], OUT["counties"] = len(d), d.County.nunique()
OUT["date_min"], OUT["date_max"] = str(d.Date.min().date()), str(d.Date.max().date())
m = d.Date.dt.month
d["Season"] = np.select([m.isin([12, 1, 2]), m.isin([3, 4, 5]), m.isin([6, 7, 8])],
                        ["Winter", "Spring", "Summer"], "Fall")
print(f"{OUT['rows']:,} county-days | {OUT['counties']} counties | {OUT['date_min']} to {OUT['date_max']}")
d.head()

## 3. Descriptive results

Mean, median, standard deviation and 90th percentile of daily AQI by season and by county, with Kruskal-Wallis tests (they do not assume a normal distribution). The two figures after the tables are extra views that are not on the poster.

In [ ]:
# Descriptive statistics
def p90(x): return x.quantile(.9)
seas = d.groupby("Season").AQI.agg(["mean", "median", "std", p90, "count"]).reindex(["Winter", "Spring", "Summer", "Fall"])
cty = d.groupby("County").AQI.agg(["mean", "median", "std", p90, "count"])
cty_big = cty[cty["count"] >= 365].sort_values("mean")
OUT["season"] = seas.round(1).reset_index().to_dict("records")
OUT["county_top"] = cty_big.tail(5).round(1).reset_index().to_dict("records")
OUT["county_bot"] = cty_big.head(5).round(1).reset_index().to_dict("records")
OUT["n_counties_365"] = len(cty_big)
OUT["kw_season"] = stats.kruskal(*[g.AQI for _, g in d.groupby("Season")]).pvalue
OUT["kw_county"] = stats.kruskal(*[g.AQI for c, g in d.groupby("County") if c in cty_big.index]).pvalue
OUT["pct_good"] = (d.Category == "Good").mean() * 100
OUT["param_by_season"] = (pd.crosstab(d.Season, d["Defining Parameter"], normalize="index") * 100).round(1).to_dict()
OUT["top_days"] = d.groupby("Date").AQI.mean().sort_values().tail(5).round(1).to_dict()
OUT["top_days"] = {str(k.date()): v for k, v in OUT["top_days"].items()}
OUT["usg_plus_by_season"] = d[d.AQI > 100].Season.value_counts().to_dict()

print("AQI by season"); print(seas.round(1).to_string())
print("\nHighest-mean counties (at least 365 days of data)"); print(cty_big.tail(5).round(1).to_string())
print("\nLowest-mean counties"); print(cty_big.head(5).round(1).to_string())
p_txt = lambda p: "< 0.001" if p < 0.001 else f"= {p:.3f}"
print(f"\nKruskal-Wallis tests: season p {p_txt(OUT['kw_season'])}, county p {p_txt(OUT['kw_county'])}")
print(f"Share of 'Good' days: {OUT['pct_good']:.1f}% | maximum AQI: {d.AQI.max()}")

In [ ]:
import matplotlib.dates as mdates

# Extra figure 1: monthly mean AQI (statewide, highest and lowest county) and season boxplot
fig, (a1, a2) = plt.subplots(1, 2, figsize=(20, 7.5), gridspec_kw={"width_ratios": [1.9, 1]})
mon = (d.set_index("Date").groupby("County").AQI.resample("MS").agg(["mean", "count"])
       .reset_index().rename(columns={"mean": "AQI"}))
mon = mon[mon["count"] >= 10]          # drop the one-day stub month (May 31, 2026)
state = d.set_index("Date").AQI.resample("MS").mean()
hi, lo = cty_big.index[-1], cty_big.index[0]
for c, col, lw in [(hi, RED, 3), (lo, GRY, 3)]:
    s = mon[mon.County == c]
    a1.plot(s.Date, s.AQI, color=col, lw=lw, label=f"{c} (highest mean)" if c == hi else f"{c} (lowest mean)")
a1.plot(state.index, state.values, color=BLK, lw=4.5, label="Ohio, all counties")
a1.set_ylabel("Monthly mean AQI"); a1.set_title("Monthly AQI, 2022 to May 2026", loc="left")
a1.legend(frameon=False, fontsize=18, loc="upper right"); a1.grid(axis="y", alpha=.3)
a1.xaxis.set_major_locator(mdates.YearLocator()); a1.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
order = ["Winter", "Spring", "Summer", "Fall"]
bp = a2.boxplot([d.loc[d.Season == s, "AQI"] for s in order], showfliers=False,
                patch_artist=True, widths=.6, medianprops=dict(color="white", lw=3))
a2.set_xticks(range(1, len(order) + 1)); a2.set_xticklabels(order)
for b, s in zip(bp["boxes"], order): b.set_facecolor(RED if s == "Summer" else BLK)
a2.set_title("Daily AQI by season", loc="left"); a2.set_ylabel("Daily AQI"); a2.grid(axis="y", alpha=.3)
fig.tight_layout(); fig.savefig("explore_monthly_season.png", dpi=300); plt.show()

# Extra figure 2: county x month heatmap (counties with >=365 days)
hm = d[d.County.isin(cty_big.index)].assign(Mon=d.Date.dt.month).pivot_table(index="County", columns="Mon", values="AQI", aggfunc="mean")
hm = hm.loc[cty_big.index[::-1]]
fig, ax = plt.subplots(figsize=(9, 13))
im = ax.imshow(hm.values, aspect="auto", cmap="Reds")
ax.set_xticks(range(12)); ax.set_xticklabels(list("JFMAMJJASOND"))
ax.set_yticks(range(len(hm))); ax.set_yticklabels(hm.index, fontsize=13)
ax.set_title("Mean AQI by county and month", loc="left", fontsize=20)
cb = fig.colorbar(im, ax=ax, fraction=.05, pad=.02); cb.set_label("Mean AQI")
for s in ax.spines.values(): s.set_visible(False)
fig.tight_layout(); fig.savefig("explore_heatmap.png", dpi=300); plt.show()

## 4. Predictive models

**Task:** predict tomorrow's AQI for each county from today's AQI, recent AQI history (lags and rolling means), weather and calendar features.

**Split:** chronological. Train on days before 2025 and test on 2025 onward, so the models never see the future.

In [ ]:
# Features, target and the chronological train/test split
W = ["Temp Max (°F)", "Temp Mean (°F)", "Precipitation (in)", "Wind Speed Max (mph)", "Wind Gust Max (mph)",
     "Relative Humidity (%)", "Solar Radiation (MJ/m²)"]
g = d.groupby("County")
d["gap_next"] = (g.Date.shift(-1) - d.Date).dt.days
d["target"] = g.AQI.shift(-1)
d["lag0"] = d.AQI
for k in [1, 2, 6]:
    d[f"lag{k}"] = g.AQI.shift(k)
d["roll3"] = g.AQI.transform(lambda s: s.rolling(3).mean())
d["roll7"] = g.AQI.transform(lambda s: s.rolling(7).mean())
d["gap_prev"] = (d.Date - g.Date.shift(1)).dt.days
wd = np.deg2rad(d["Wind Direction (°)"])
d["wind_sin"], d["wind_cos"] = np.sin(wd), np.cos(wd)
d["doy_sin"], d["doy_cos"] = np.sin(2 * np.pi * d.Date.dt.dayofyear / 365.25), np.cos(2 * np.pi * d.Date.dt.dayofyear / 365.25)
d["dow_next"] = (d.Date + pd.Timedelta(days=1)).dt.dayofweek
d["ozone_def"] = (d["Defining Parameter"] == "Ozone").astype(int)
d["county_mean"] = 0.0
FEATS = ["lag0", "lag1", "lag2", "lag6", "roll3", "roll7", "doy_sin", "doy_cos", "dow_next", "ozone_def",
         "wind_sin", "wind_cos", "county_mean"] + W
m = d.dropna(subset=["target", "lag6", "roll7"])
m = m[(m.gap_next == 1)].copy()
OUT["model_rows"] = len(m)
train = m[m.Date < "2025-01-01"].copy(); test = m[m.Date >= "2025-01-01"].copy()
cm = train.groupby("County").target.mean(); gm = train.target.mean()
for df in (train, test): df["county_mean"] = df.County.map(cm).fillna(gm)
OUT["n_train"], OUT["n_test"] = len(train), len(test)
OUT["train_range"] = [str(train.Date.min().date()), str(train.Date.max().date())]
OUT["test_range"] = [str(test.Date.min().date()), str(test.Date.max().date())]
Xtr, ytr, Xte, yte = train[FEATS], train.target, test[FEATS], test.target

print(f"Rows used for modelling: {OUT['model_rows']:,}")
print(f"Train: {OUT['n_train']:,} rows ({OUT['n_train'] / OUT['model_rows']:.0%}), {OUT['train_range'][0]} to {OUT['train_range'][1]}")
print(f"Test:  {OUT['n_test']:,} rows ({OUT['n_test'] / OUT['model_rows']:.0%}), {OUT['test_range'][0]} to {OUT['test_range'][1]}")

### Baseline and four models

Persistence baseline (tomorrow = today), Ridge regression, two Random Forests (A: default settings, B: settings chosen by hand) and Hist Gradient Boosting. Random Forest is bagging and Hist Gradient Boosting is boosting, the two ensemble ideas from the proposal.

In [ ]:
def ev(name, pred, extra=""):
    r = dict(model=name, MAE=mean_absolute_error(yte, pred), RMSE=mean_squared_error(yte, pred) ** .5,
             R2=r2_score(yte, pred), note=extra)
    print(r); return r

res = [ev("Persistence baseline", test.lag0)]
models = {
    "Ridge Regression": make_pipeline(StandardScaler(), Ridge(alpha=1.0)),
    "Random Forest A": RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    "Random Forest B": RandomForestRegressor(n_estimators=300, max_depth=12, min_samples_leaf=10, max_features=0.5, random_state=42, n_jobs=-1),
    "Hist Gradient Boosting": HistGradientBoostingRegressor(random_state=42),
}
fitted = {}
for n, mdl in models.items():
    mdl.fit(Xtr, ytr); fitted[n] = mdl; res.append(ev(n, mdl.predict(Xte)))

best_untuned = min(res[1:], key=lambda r: r["RMSE"])["model"]
OUT["best_untuned"] = best_untuned

pd.DataFrame(res)[["model", "MAE", "RMSE", "R2"]].round(3)

### Hyperparameter tuning (the slow step)

The best untuned model family is tuned with `RandomizedSearchCV` (25 random settings, 4 folds in time order). The rows are sorted by date, there is a 10-day gap between each fold's training and validation parts, and the county average is learned inside each fold, so nothing from the validation days leaks into training.

In [ ]:
# ---- hyperparameter tuning: chronological, leak-free cross-validation ----
# The rows are ordered by county, so a plain TimeSeriesSplit would cut by COUNTY blocks, not by time.
# Fix: (1) sort the training rows by date, (2) leave a ~10-day gap (300 rows) between each fold's train and
# validation part, (3) learn the county-mean feature inside each fold (never from the validation rows).
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline

class CountyMean(BaseEstimator, TransformerMixin):
    def fit(self, X, y):
        self.map_ = pd.Series(np.asarray(y)).groupby(X["County"].to_numpy()).mean()
        self.global_ = float(np.mean(y))
        return self
    def transform(self, X):
        X = X.copy()
        X["county_mean"] = X["County"].map(self.map_).fillna(self.global_).to_numpy()
        return X.drop(columns=["County"])

FEATS_CV = [f for f in FEATS if f != "county_mean"] + ["County"]
train_cv = train.sort_values(["Date", "County"]).reset_index(drop=True)
Xtr_cv, ytr_cv, Xte_cv = train_cv[FEATS_CV], train_cv.target, test[FEATS_CV]
tscv = TimeSeriesSplit(n_splits=4, gap=300)
OUT["cv_folds"] = [[str(train_cv.Date.iloc[a].min().date()), str(train_cv.Date.iloc[a].max().date()),
                    str(train_cv.Date.iloc[b].min().date()), str(train_cv.Date.iloc[b].max().date())] for a, b in tscv.split(Xtr_cv)]
print("CV folds (train from, train to, validate from, validate to):", OUT["cv_folds"])
if best_untuned == "Hist Gradient Boosting":
    est = HistGradientBoostingRegressor(random_state=42)
    grid = {"learning_rate": [.02, .05, .1], "max_iter": [200, 400, 800], "max_leaf_nodes": [15, 31, 63],
            "min_samples_leaf": [20, 50, 100], "l2_regularization": [0, .1, 1.0], "loss": ["squared_error", "absolute_error"]}
else:
    est = RandomForestRegressor(random_state=42, n_jobs=-1)
    grid = {"n_estimators": [200, 400], "max_depth": [8, 12, 16, None], "min_samples_leaf": [5, 10, 20, 40],
            "max_features": [.3, .5, .7]}
base = Pipeline([("cm", CountyMean()), ("model", est)])
grid = {"model__" + k: v for k, v in grid.items()}
search = RandomizedSearchCV(base, grid, n_iter=25, cv=tscv, scoring="neg_root_mean_squared_error", random_state=42, n_jobs=-1)
search.fit(Xtr_cv, ytr_cv)
OUT["best_params"] = {k.replace("model__", ""): v for k, v in search.best_params_.items()}; OUT["cv_rmse"] = -search.best_score_
tuned = search.best_estimator_
tname = f"Tuned {best_untuned}"
res.append(ev(tname, tuned.predict(Xte_cv)))
OUT["results"] = res

print("Best settings:", OUT["best_params"])
print(f"Cross-validated RMSE of the best setting: {OUT['cv_rmse']:.2f}")
pd.DataFrame(res)[["model", "MAE", "RMSE", "R2"]].round(3)

### What drives the predictions, and how it does on AQI categories

Permutation importance shows how much the test error grows when one feature is shuffled. The categories are Good (0-50), Moderate (51-100) and USG+ (above 100, unhealthy for sensitive groups or worse).

In [ ]:
# importance (permutation) for tuned model
from sklearn.inspection import permutation_importance
pi = permutation_importance(tuned, Xte_cv, yte, n_repeats=5, random_state=0, scoring="neg_root_mean_squared_error", n_jobs=-1)
imp = pd.Series(pi.importances_mean, index=FEATS_CV).sort_values()
OUT["importance"] = imp.round(3).to_dict()
pt = tuned.predict(Xte_cv); test["pred"] = pt
cat = lambda a: pd.cut(a, [-1, 50, 100, 1000], labels=["Good", "Moderate", "USG+"])
cmx = pd.crosstab(cat(yte), cat(pd.Series(pt, index=yte.index)))
OUT["cat_confusion"] = cmx.to_dict()
OUT["cat_acc"] = float((cat(yte) == cat(pd.Series(pt, index=yte.index))).mean())
OUT["cat_acc_persist"] = float((cat(yte) == cat(test.lag0)).mean())
hi_mask = yte > 100
OUT["mae_hi_tuned"] = mean_absolute_error(yte[hi_mask], pt[hi_mask]); OUT["n_hi"] = int(hi_mask.sum())
OUT["mae_hi_persist"] = mean_absolute_error(yte[hi_mask], test.lag0[hi_mask])
OUT["mae_by_season"] = test.assign(e=(test.pred - test.target).abs()).groupby("Season").e.mean().round(2).to_dict()

print("Top 6 features (RMSE increase when the feature is shuffled):"); print(imp.tail(6)[::-1].round(3).to_string())
print("\nConfusion matrix (rows = actual category, columns = predicted):"); print(cmx.to_string())
print(f"\nTest days with actual AQI over 100: {OUT['n_hi']}")

In [ ]:
# Extra figure 3: importance + actual vs predicted
lab = {"lag0": "AQI today", "lag1": "AQI yesterday", "lag2": "AQI 2 days ago", "lag6": "AQI 6 days ago",
       "roll3": "3-day mean AQI", "roll7": "7-day mean AQI", "doy_sin": "Day of year (sin)", "doy_cos": "Day of year (cos)",
       "dow_next": "Day of week", "ozone_def": "Ozone is defining", "wind_sin": "Wind dir (sin)", "wind_cos": "Wind dir (cos)",
       "county_mean": "County mean AQI", "County": "County", "Temp Max (°F)": "Max temp", "Temp Mean (°F)": "Mean temp",
       "Precipitation (in)": "Precipitation", "Wind Speed Max (mph)": "Max wind speed", "Wind Gust Max (mph)": "Max wind gust",
       "Relative Humidity (%)": "Rel. humidity", "Solar Radiation (MJ/m²)": "Solar radiation"}
top = imp.tail(8)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(20, 8), gridspec_kw={"width_ratios": [1, 1]})
a1.barh([lab[i] for i in top.index], top.values, color=[RED if v == top.max() else BLK for v in top.values])
a1.set_xlabel("RMSE increase when shuffled"); a1.set_title("Top features (permutation)", loc="left")
s = test[(test.County == hi) & (test.Date >= "2025-05-01") & (test.Date < "2025-09-01")]
a2.plot(s.Date + pd.Timedelta(days=1), s.target, color=BLK, lw=3, label="Actual")
a2.plot(s.Date + pd.Timedelta(days=1), s.pred, color=RED, lw=3, label="Predicted")
a2.set_title(f"{hi} County, summer 2025", loc="left"); a2.set_ylabel("Next-day AQI")
a2.legend(frameon=False, fontsize=18); a2.grid(axis="y", alpha=.3)
import matplotlib.dates as mdates
a2.xaxis.set_major_locator(mdates.MonthLocator()); a2.xaxis.set_major_formatter(mdates.DateFormatter("%b"))
fig.tight_layout(); fig.savefig("explore_pred.png", dpi=300); plt.show()

In [ ]:
# Save every number computed above
with open("results.json", "w") as f:
    json.dump(OUT, f, indent=1, default=lambda o: float(o) if isinstance(o, (np.floating, np.integer)) else str(o))
print("Saved results.json")

## 5. Poster figures

Redraws the two figures that are on the poster: `fig_desc.png` (monthly AQI and AQI by season) and `fig_imp.png` (top features). They read `results.json` from the step above.

In [ ]:
import matplotlib.dates as mdates

RED, BLK, GRY = "#C8102E", "#1A1A1A", "#9A9A9A"
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 20, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.titleweight": "bold", "axes.titlesize": 22})
with open("results.json") as f:
    o = json.load(f)
raw = pd.read_excel(DATA).rename(columns={"county Name": "County"})   # own copy, so the analysis table stays untouched
mo = raw.Date.dt.month
raw["Season"] = np.select([mo.isin([12, 1, 2]), mo.isin([3, 4, 5]), mo.isin([6, 7, 8])], ["Winter", "Spring", "Summer"], "Fall")

# ---------- Descriptive figure (placed 10.6 in wide) ----------
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10.6, 5.2), gridspec_kw={"width_ratios": [1.8, 1]})
mon = (raw.set_index("Date").groupby("County").AQI.resample("MS").agg(["mean", "count"])
       .reset_index().rename(columns={"mean": "AQI"}))
mon = mon[mon["count"] >= 10]          # drop the one-day stub month (May 31 2026 only has 1 obs for some counties)
state = raw.set_index("Date").AQI.resample("MS").mean()
for c, col, lab in [("Cuyahoga", RED, "Cuyahoga"), ("Belmont", GRY, "Belmont")]:
    s = mon[mon.County == c]; a1.plot(s.Date, s.AQI, color=col, lw=2.6, label=lab)
a1.plot(state.index, state.values, color=BLK, lw=3.6, label="Ohio avg.")
a1.annotate("Wildfire smoke,\nJune 2023", xy=(pd.Timestamp("2023-06-01"), state["2023-06-01"]),
            xytext=(pd.Timestamp("2023-12-01"), 84), fontsize=17, arrowprops=dict(arrowstyle="->", color=BLK, lw=2))
a1.set_ylim(-9, 106); a1.set_yticks([20, 40, 60, 80, 100]); a1.set_ylabel("Monthly mean AQI")
a1.set_title("Monthly AQI", loc="left")
a1.legend(frameon=False, fontsize=17, loc="lower center", ncol=3, handlelength=1.1, columnspacing=1.0,
          bbox_to_anchor=(0.5, -0.02))
a1.xaxis.set_major_locator(mdates.YearLocator()); a1.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
a1.grid(axis="y", alpha=.3)
order = ["Winter", "Spring", "Summer", "Fall"]
bp = a2.boxplot([raw.loc[raw.Season == s, "AQI"] for s in order], showfliers=False,
                patch_artist=True, widths=.6, medianprops=dict(color="white", lw=3))
a2.set_xticks(range(1, 5)); a2.set_xticklabels(["Win", "Spr", "Sum", "Fall"])
for b, s in zip(bp["boxes"], order): b.set_facecolor(RED if s == "Summer" else BLK)
a2.set_title("AQI by season", loc="left"); a2.set_ylabel("Daily AQI"); a2.grid(axis="y", alpha=.3)
fig.tight_layout(); fig.savefig("fig_desc.png", dpi=300); plt.show()

# ---------- Predictive figure (placed ~5.2 in wide, next to the table) ----------
lab = {"lag0": "AQI today", "Wind Gust Max (mph)": "Max wind gust", "doy_cos": "Day of year", "county_mean": "County mean",
       "Solar Radiation (MJ/m²)": "Solar rad.", "Temp Mean (°F)": "Mean temp", "Wind Speed Max (mph)": "Max wind",
       "Precipitation (in)": "Precip.", "County": "County", "roll7": "7-day mean", "roll3": "3-day mean"}
imp = pd.Series(o["importance"]).sort_values().tail(6)
fig, ax = plt.subplots(figsize=(5.2, 4.7))
ax.barh([lab.get(i, i) for i in imp.index], imp.values, color=[RED if v == imp.max() else BLK for v in imp.values], height=.7)
for y, v in enumerate(imp.values): ax.text(v + 0.06, y, f"{v:.2f}", va="center", fontsize=18)
ax.set_xlim(0, 4.0); ax.set_xticks([0, 1, 2, 3])
ax.set_xlabel("Importance (RMSE rise)", fontsize=19); ax.set_title("Top 6 features", loc="left", fontsize=22)
ax.tick_params(labelsize=19)
fig.tight_layout(); fig.savefig("fig_imp.png", dpi=300); plt.show()

## 6. Download

Zips `results.json` and all figures into `aqi_outputs.zip` and downloads it. If the browser blocks the download, open the folder icon on the left side of Colab and download the zip from there.

In [ ]:
import glob
from zipfile import ZipFile

outputs = ["results.json", "fig_desc.png", "fig_imp.png"] + sorted(glob.glob("explore_*.png"))
with ZipFile("aqi_outputs.zip", "w") as z:
    for name in outputs:
        z.write(name)
print("Zipped:", ", ".join(outputs))

try:
    from google.colab import files
    files.download("aqi_outputs.zip")
except ImportError:
    print("Not running in Colab: aqi_outputs.zip is in the current folder.")